# Pairwise comparison of the Imaging, Blood and Hybrid Models (DeLong and McNemar tests)

The three models were evaluated on the same 58 test patients (30 with acute cholecystitis, 28 controls) of the 7:3 split.
Their AUCs are compared pairwise with the DeLong test for two correlated ROC curves (DeLong et al., Biometrics 1988;
fast algorithm of Sun and Xu, IEEE Signal Process Lett 2014). P-values are adjusted for the three comparisons with the
Holm method. As a check, 95% CIs of the AUC differences are also estimated by paired stratified bootstrap (2,000 resamples).
Accuracy, sensitivity and specificity at the threshold of 0.5 are compared with the exact McNemar test, with Holm adjustment
for the three comparisons of each metric.

All-slice evaluations of the Imaging and Hybrid Models use the same slices, obtained with the gallbladder masks of
TotalSegmentator run without `--roi_subset` and without `--fast`.

**Patient-level scores**
- Imaging Model: maximum predicted probability over all slices containing the gallbladder.
- Blood Model: predicted probability (one prediction per patient).
- Hybrid Model: maximum predicted probability over all slices containing the gallbladder.

**Input files**
- `img_path_all_slices_test_20260702_full_noroi_with_pred.csv`: Imaging Model (slice level)
- `blood_model_test_pred.csv`: Blood Model
- `hybrid_model_all_slices_test_pred.csv`: Hybrid Model (slice level)

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

DATA_DIR = "."
EXCLUDE_PATIENT_IDS = [87, 97]
N_BOOT = 2000
SEED = 42

In [2]:
# ===== Patient-level scores of the three models =====
image = (
    pd.read_csv(f"{DATA_DIR}/img_path_all_slices_test_20260702_full_noroi_with_pred.csv")
    .groupby("patient_id")
    .agg(label=("label", "first"), image=("y_prob", "max"))
    .reset_index()
)

blood = pd.read_csv(f"{DATA_DIR}/blood_model_test_pred.csv")
path_to_id = pd.read_csv(f"{DATA_DIR}/img_path_and_clinical_info_remove_nan_img_feature_0408_remove_dup.csv")[["path", "patient_id"]]
blood = blood.merge(path_to_id, on="path", how="left")
assert blood["patient_id"].notna().all()
blood = blood[["patient_id", "label", "prob"]].rename(columns={"prob": "blood", "label": "label_blood"})

hybrid = (
    pd.read_csv(f"{DATA_DIR}/hybrid_model_all_slices_test_pred.csv")
    .groupby("patient_id")
    .agg(label_hybrid=("label", "first"), hybrid=("prediction_score_1", "max"))
    .reset_index()
)

df = (
    image
    .merge(blood, on="patient_id")
    .merge(hybrid, on="patient_id")
)
df = df[~df["patient_id"].isin(EXCLUDE_PATIENT_IDS)].reset_index(drop=True)
assert (df["label"] == df["label_blood"]).all() and (df["label"] == df["label_hybrid"]).all()
df = df.drop(columns=["label_blood", "label_hybrid"])
df.to_csv(f"{DATA_DIR}/delong_patient_level_scores.csv", index=False)

score_cols = {
    "Imaging Model": "image",
    "Blood Model": "blood",
    "Hybrid Model": "hybrid",
}
print("patients:", len(df), df["label"].value_counts().to_dict())
pd.Series({name: round(roc_auc_score(df["label"], df[col]), 3) for name, col in score_cols.items()}, name="AUC")

patients: 58 {1: 30, 0: 28}


Imaging Model    0.850
Blood Model      0.946
Hybrid Model     0.996
Name: AUC, dtype: float64

In [3]:
# ===== DeLong test for two correlated AUCs (fast algorithm of Sun & Xu, 2014) =====
def midrank(x):
    order = np.argsort(x)
    xs = x[order]
    n = len(x)
    ranks = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and xs[j] == xs[i]:
            j += 1
        ranks[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n)
    out[order] = ranks
    return out

def delong_paired(y_true, score_a, score_b):
    # Returns AUC_a, AUC_b, difference, its SE, 95% CI and two-sided p-value
    y_true = np.asarray(y_true).astype(int)
    scores = np.vstack([score_a, score_b]).astype(float)
    pos, neg = scores[:, y_true == 1], scores[:, y_true == 0]
    m, n = pos.shape[1], neg.shape[1]
    tx = np.array([midrank(p) for p in pos])
    ty = np.array([midrank(q) for q in neg])
    tz = np.array([midrank(s) for s in np.hstack([pos, neg])])
    aucs = (tz[:, :m].sum(axis=1) / m - (m + 1) / 2) / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    diff = aucs[0] - aucs[1]
    se = np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    z = diff / se
    p = 2 * stats.norm.sf(abs(z))
    return aucs[0], aucs[1], diff, se, diff - 1.96 * se, diff + 1.96 * se, p

def bootstrap_diff_ci(y_true, score_a, score_b, n_boot=N_BOOT, seed=SEED):
    # Paired stratified bootstrap: the same resampled patients are used for both models
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    pos, neg = np.where(y_true == 1)[0], np.where(y_true == 0)[0]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        diffs.append(roc_auc_score(y_true[idx], score_a[idx]) - roc_auc_score(y_true[idx], score_b[idx]))
    return np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

def holm(pvals):
    pvals = np.asarray(pvals)
    order = np.argsort(pvals)
    adj = np.empty(len(pvals))
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(pvals) - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj

In [4]:
# ===== Pairwise comparisons =====
def compare(pairs):
    rows = []
    for a, b in pairs:
        auc_a, auc_b, diff, se, lo, hi, p = delong_paired(df["label"], df[score_cols[a]], df[score_cols[b]])
        b_lo, b_hi = bootstrap_diff_ci(df["label"].values, df[score_cols[a]].values, df[score_cols[b]].values)
        rows.append({
            "comparison": f"{a} vs {b}",
            "AUC 1": round(auc_a, 3), "AUC 2": round(auc_b, 3),
            "difference (95% CI, DeLong)": f"{diff:.3f} ({lo:.3f} to {hi:.3f})",
            "difference (95% CI, bootstrap)": f"{diff:.3f} ({b_lo:.3f} to {b_hi:.3f})",
            "p (DeLong)": p,
        })
    out = pd.DataFrame(rows)
    out["p (Holm-adjusted)"] = holm(out["p (DeLong)"])
    for col in ["p (DeLong)", "p (Holm-adjusted)"]:
        out[col] = out[col].map(lambda v: "<0.001" if v < 0.001 else f"{v:.3f}")
    return out

pd.set_option("display.max_colwidth", None)

results = compare([
    ("Hybrid Model", "Imaging Model"),
    ("Hybrid Model", "Blood Model"),
    ("Blood Model", "Imaging Model"),
])
results.to_csv(f"{DATA_DIR}/delong_main.csv", index=False)
display(results)

,comparison,AUC 1,AUC 2,"difference (95% CI, DeLong)","difference (95% CI, bootstrap)",p (DeLong),p (Holm-adjusted)
0,Hybrid Model vs Imaging Model,0.996,0.850,0.146 (0.045 to 0.248),0.146 (0.052 to 0.256),0.005,0.014
1,Hybrid Model vs Blood Model,0.996,0.946,0.051 (-0.006 to 0.107),0.051 (0.006 to 0.115),0.078,0.156
2,Blood Model vs Imaging Model,0.946,0.850,0.096 (-0.031 to 0.222),0.096 (-0.031 to 0.229),0.137,0.156


## Accuracy, sensitivity and specificity (exact McNemar test)

For each pair of models, the patients classified correctly by one model but not by the other (discordant pairs) are counted, and the two-sided exact McNemar test (binomial test on the discordant pairs) is applied. Sensitivity is compared in the patients with acute cholecystitis and specificity in the control patients. P-values are adjusted with the Holm method within each metric.

In [5]:
from scipy.stats import binomtest

THRESHOLD = 0.5
y = df["label"].values.astype(int)
correct = {name: ((df[col].values >= THRESHOLD).astype(int) == y) for name, col in score_cols.items()}

def mcnemar_exact(correct_a, correct_b):
    n10 = int((correct_a & ~correct_b).sum())   # correct by model 1 only
    n01 = int((~correct_a & correct_b).sum())   # correct by model 2 only
    p = 1.0 if n10 + n01 == 0 else binomtest(n10, n10 + n01, 0.5).pvalue
    return n10, n01, p

pairs = [("Hybrid Model", "Imaging Model"), ("Hybrid Model", "Blood Model"), ("Blood Model", "Imaging Model")]
subsets = {"Accuracy": np.ones_like(y, dtype=bool), "Sensitivity": y == 1, "Specificity": y == 0}

rows = []
for metric, mask in subsets.items():
    block = []
    for a, b in pairs:
        ca, cb = correct[a][mask], correct[b][mask]
        n10, n01, p = mcnemar_exact(ca, cb)
        block.append({
            "metric": metric, "n": int(mask.sum()), "comparison": f"{a} vs {b}",
            "value 1": round(ca.mean(), 3), "value 2": round(cb.mean(), 3),
            "correct by 1 only / by 2 only": f"{n10} / {n01}", "p": p,
        })
    adj = holm([r["p"] for r in block])
    for r, pa in zip(block, adj):
        r["p (Holm-adjusted)"] = pa
    rows += block

mcnemar_results = pd.DataFrame(rows)
for col in ["p", "p (Holm-adjusted)"]:
    mcnemar_results[col] = mcnemar_results[col].map(lambda v: "<0.001" if v < 0.001 else f"{v:.3f}")
mcnemar_results.to_csv(f"{DATA_DIR}/mcnemar_accuracy_sensitivity_specificity.csv", index=False)
display(mcnemar_results)

,metric,n,comparison,value 1,value 2,correct by 1 only / by 2 only,p,p (Holm-adjusted)
0,Accuracy,58,Hybrid Model vs Imaging Model,0.931,0.707,13 / 0,<0.001,<0.001
1,Accuracy,58,Hybrid Model vs Blood Model,0.931,0.914,5 / 4,1.000,1.000
2,Accuracy,58,Blood Model vs Imaging Model,0.914,0.707,16 / 4,0.012,0.024
3,Sensitivity,30,Hybrid Model vs Imaging Model,1.000,0.833,5 / 0,0.062,0.188
4,Sensitivity,30,Hybrid Model vs Blood Model,1.000,0.900,3 / 0,0.250,0.500
5,Sensitivity,30,Blood Model vs Imaging Model,0.900,0.833,5 / 3,0.727,0.727
6,Specificity,28,Hybrid Model vs Imaging Model,0.857,0.571,8 / 0,0.008,0.019
7,Specificity,28,Hybrid Model vs Blood Model,0.857,0.929,2 / 4,0.688,0.688
8,Specificity,28,Blood Model vs Imaging Model,0.929,0.571,11 / 1,0.006,0.019


## Summary of results

Test set: 58 patients (30 with acute cholecystitis, 28 controls). All three models were evaluated on the same patients.

**AUC of each model:** Imaging Model 0.850, Blood Model 0.946, Hybrid Model 0.996.

### AUC (DeLong test)

| Comparison | AUC | Difference (95% CI) | p | p (Holm-adjusted) |
|---|---|---|---|---|
| Hybrid Model vs Imaging Model | 0.996 vs 0.850 | 0.146 (0.045 to 0.248) | 0.005 | 0.014 |
| Hybrid Model vs Blood Model | 0.996 vs 0.946 | 0.051 (-0.006 to 0.107) | 0.078 | 0.156 |
| Blood Model vs Imaging Model | 0.946 vs 0.850 | 0.096 (-0.031 to 0.222) | 0.137 | 0.156 |

### Accuracy, sensitivity and specificity (exact McNemar test)

| Metric | Comparison | Value | Correct by 1 only / by 2 only | p | p (Holm-adjusted) |
|---|---|---|---|---|---|
| Accuracy (n = 58) | Hybrid Model vs Imaging Model | 0.931 vs 0.707 | 13 / 0 | <0.001 | <0.001 |
| Accuracy (n = 58) | Hybrid Model vs Blood Model | 0.931 vs 0.914 | 5 / 4 | 1.000 | 1.000 |
| Accuracy (n = 58) | Blood Model vs Imaging Model | 0.914 vs 0.707 | 16 / 4 | 0.012 | 0.024 |
| Sensitivity (n = 30) | Hybrid Model vs Imaging Model | 1.000 vs 0.833 | 5 / 0 | 0.062 | 0.188 |
| Sensitivity (n = 30) | Hybrid Model vs Blood Model | 1.000 vs 0.900 | 3 / 0 | 0.250 | 0.500 |
| Sensitivity (n = 30) | Blood Model vs Imaging Model | 0.900 vs 0.833 | 5 / 3 | 0.727 | 0.727 |
| Specificity (n = 28) | Hybrid Model vs Imaging Model | 0.857 vs 0.571 | 8 / 0 | 0.008 | 0.019 |
| Specificity (n = 28) | Hybrid Model vs Blood Model | 0.857 vs 0.929 | 2 / 4 | 0.688 | 0.688 |
| Specificity (n = 28) | Blood Model vs Imaging Model | 0.929 vs 0.571 | 11 / 1 | 0.006 | 0.019 |

### Interpretation

- The Hybrid Model had a significantly higher AUC than the Imaging Model (Holm-adjusted p = 0.014), with higher accuracy (Holm-adjusted p < 0.001) and specificity (p = 0.019).
- The differences between the Hybrid Model and the Blood Model were not statistically significant for any metric (AUC: Holm-adjusted p = 0.156).
- The Blood Model had higher accuracy (p = 0.024) and specificity (p = 0.019) than the Imaging Model, whereas the difference in AUC was not significant (p = 0.156).
- Sensitivity did not differ significantly between any pair of models.
- With 58 test patients, the study has limited power to detect small differences; non-significant results should not be interpreted as evidence that the models perform equally.
- Because the AUC of the Hybrid Model is close to 1, the normal approximation of the DeLong test is less accurate for the comparison with the Blood Model; the paired bootstrap CI of this difference (shown above) excludes zero, whereas the DeLong CI does not, so this comparison should be regarded as borderline.